# Estado del arte: R0 por la matriz de la siguiente generación y la derivación del tamaño final

- **Nivel:** 3, Estado del arte
- **Prerrequisitos:** cuadernos 01 y 02 de este tema
- **Tiempo de lectura:** unos 30 minutos

## Objetivos

Al terminar este cuaderno puedes:

- calcular $R_0$ como el radio espectral de la matriz de la siguiente generación y compararlo con $\beta/\gamma$;
- derivar la relación del tamaño final desde las ecuaciones del modelo, incluida la infección inicial, y comprobar la derivación sobre la trayectoria del motor;
- explicar la diferencia entre la tasa de ataque del motor y la relación del cuaderno 02, y quitarla extrapolando en el paso.

## Preparación

El entorno de los cuadernos está fijado en `notebooks/requirements.txt` y se construye con `scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` en Windows). Los paquetes que usa este cuaderno, con sus versiones exactas:

```bash
pip install numpy==2.4.6 matplotlib==3.11.2
```

Este producto de ejemplo no publica un paquete propio: su método vive en `data-pipeline/pipeline/`, así que el cuaderno importa ese código por su ruta. Un producto importa en cambio su propio paquete de PyPI (`pip install <paquete>==<versión>`).

La siguiente celda importa las herramientas e imprime sus versiones.

In [1]:
# Imprimir las versiones de las herramientas de este cuaderno, para compararlas con notebooks/requirements.txt
import importlib.metadata as md
import json

import numpy as np
import matplotlib.pyplot as plt

for name in ("numpy", "matplotlib"):
    print(name, md.version(name))

numpy 2.4.6
matplotlib 3.11.2


El motor se importa desde `data-pipeline/` por su ruta, como en el tema 01.

In [2]:
# El cuaderno está tres carpetas bajo la raíz del repositorio; los módulos del pipeline se importan por su ruta
import sys
from pathlib import Path

ROOT = Path("../../..").resolve()
sys.path.insert(0, str(ROOT / "data-pipeline"))
from pipeline.model.sir import simulate
from pipeline.io.schema import SIRParams
from pipeline.oracles import final_size

print(simulate.__module__, SIRParams.__module__)

pipeline.model.sir pipeline.io.schema


## Datos

Los parámetros de los cinco casos calculados, los mismos de los cuadernos anteriores, de modo que cada comparación de abajo sea sobre los mismos casos que el nivel 2.

- Propietario: este repositorio (el ejemplo de la plantilla de producto), https://github.com/fsantibanezleal/CAOS_PRODUCT_TEMPLATE.
- Clase de licencia: datos sintéticos que el pipeline produce desde los parámetros de cada manifiesto (`real_or_synthetic` es `synthetic`); no hay conjunto de datos de terceros, así que no aplica licencia de datos, y la licencia MIT del repositorio cubre los artefactos.
- Archivos leídos: `data/derived/manifests/index.json` y `data/derived/manifests/<case>.json`.

La siguiente celda define los lectores y lista los casos.

In [3]:
# Dos lectores de los artefactos versionados: un archivo ausente detiene el cuaderno y nombra su ruta
def read_json(relative):
    return json.loads((ROOT / "data" / "derived" / relative).read_text(encoding="utf-8"))


def params_of(case_id):
    p = read_json(f"manifests/{case_id}.json")["params"]
    return SIRParams(case_id, p["beta"], p["gamma"], p["N"], p["I0"], p["days"])


CASES = [c["case_id"] for c in read_json("manifests/index.json")["cases"]]
OUTBREAKS = [c for c in CASES if c.startswith("EX")]
SUPERCRITICAL = [c for c in OUTBREAKS if params_of(c).beta > params_of(c).gamma]
print(CASES)
print("casos que pueden despegar:", SUPERCRITICAL)

['CTRL_degenerate', 'EX01_subcritical', 'EX02_epidemic', 'EX03_fast_burn', 'EX04_slow_spread']
casos que pueden despegar: ['EX02_epidemic', 'EX03_fast_burn', 'EX04_slow_spread']


## Método

Lo que estableció el nivel 2, en dos líneas: $R_0 = \beta / \gamma$ (cuaderno 02 del tema 01) y el tamaño final $z$ resuelve $z = 1 - e^{-R_0 z}$ (cuaderno 02 de este tema), contrastado con la corrida de Euler hacia adelante del motor con paso de un cuarto de día. Este cuaderno pregunta si el estado del arte da las mismas dos respuestas y qué agrega.

## Estado del arte

### R0 por la matriz de la siguiente generación

Para un modelo compartimental, van den Driessche y Watmough (2002) separan las ecuaciones de los compartimentos infectados en la tasa de nuevas infecciones $\mathcal{F}$ y la tasa de transiciones $\mathcal{V}$ (recuperación, muerte), linealizan ambas en el estado libre de enfermedad hasta las matrices $F$ y $V$, y definen $R_0$ como el radio espectral de la matriz de la siguiente generación $F V^{-1}$. Su condición es que el estado libre de enfermedad es estable cuando $R_0 < 1$. Para el modelo SIR hay un compartimento infectado, así que $F = [\beta]$ y $V = [\gamma]$ son de $1 \times 1$ y $F V^{-1} = [\beta / \gamma]$. La celda lo calcula como el mayor valor propio en valor absoluto, igual que para un modelo mayor, y también la tasa de crecimiento del compartimento infectado en el estado libre de enfermedad, el valor propio de $F - V$.

In [4]:
# R0 como radio espectral de F V^-1, y la tasa de crecimiento del compartimento infectado como valor propio de F - V
for case_id in CASES:
    p = params_of(case_id)
    F = np.array([[p.beta]])
    V = np.array([[p.gamma]])
    r0_ngm = float(np.max(np.abs(np.linalg.eigvals(F @ np.linalg.inv(V)))))
    growth = float(np.max(np.linalg.eigvals(F - V).real))
    print(f"{case_id:18s} R0 (matriz)={r0_ngm:.4f}  beta/gamma={p.beta / p.gamma:.4f}  tasa de crecimiento={growth:+.3f}  R0>1: {r0_ngm > 1}  crecimiento>0: {growth > 0}")

CTRL_degenerate    R0 (matriz)=2.0000  beta/gamma=2.0000  tasa de crecimiento=+0.200  R0>1: True  crecimiento>0: True
EX01_subcritical   R0 (matriz)=0.7200  beta/gamma=0.7200  tasa de crecimiento=-0.070  R0>1: False  crecimiento>0: False
EX02_epidemic      R0 (matriz)=2.7500  beta/gamma=2.7500  tasa de crecimiento=+0.350  R0>1: True  crecimiento>0: True
EX03_fast_burn     R0 (matriz)=6.0000  beta/gamma=6.0000  tasa de crecimiento=+1.000  R0>1: True  crecimiento>0: True
EX04_slow_spread   R0 (matriz)=1.2000  beta/gamma=1.2000  tasa de crecimiento=+0.050  R0>1: True  crecimiento>0: True


La condición de estabilidad se puede contrastar con el motor. Corre cada caso sus dos primeros días con un paso pequeño y estima la tasa de crecimiento de los infectados, $\ln(I(2)/I(0))/2$, para compararla con $\beta - \gamma$. Coinciden solo de forma aproximada, porque la reserva de susceptibles ya es un poco menor que $N$ tras dos días; la celda imprime ambas.

In [5]:
# Tasa de crecimiento de los infectados en los dos primeros días de la corrida del motor, contra beta - gamma
for case_id in OUTBREAKS:
    p = params_of(case_id)
    short = SIRParams(p.case_id, p.beta, p.gamma, p.N, p.I0, 2)
    r = simulate(short, dt=0.0625)
    measured = float(np.log(r.I[-1] / r.I[0]) / 2.0)
    print(f"{case_id:18s} medida={measured:+.4f}  beta-gamma={p.beta - p.gamma:+.4f}")

EX01_subcritical   medida=-0.0703  beta-gamma=-0.0700
EX02_epidemic      medida=+0.3458  beta-gamma=+0.3500
EX03_fast_burn     medida=+0.9662  beta-gamma=+1.0000
EX04_slow_spread   medida=+0.0497  beta-gamma=+0.0500


### El tamaño final, derivado

Miller (2012) da una derivación cerrada del tamaño final. Aquí está para el sistema SIR, desde sus ecuaciones, conservando la infección inicial. Dividir la primera por la tercera ecuación elimina el tiempo: $dS/dR = -(\beta/\gamma) S/N = -R_0 S/N$. Integrando desde el inicio, donde $R = 0$ y $S = S_0$:

$$S = S_0 \, e^{-R_0 R / N}$$

Al final del brote $I = 0$, así que $S_\infty = N - R_\infty$. Con $z = R_\infty / N$ e $i_0 = I_0 / N$, donde $S_0 = N (1 - i_0)$:

$$1 - z = (1 - i_0) \, e^{-R_0 z}$$

Cuando la infección inicial es despreciable ($i_0 \to 0$) esta es la relación del nivel 2, $z = 1 - e^{-R_0 z}$. Primero la celda comprueba el resultado intermedio sobre la trayectoria del motor: la cantidad $S \, e^{R_0 R / N}$ debería mantenerse constante. Euler hacia adelante solo la mantiene hasta su propio error, así que la celda muestra la deriva con el paso del motor y con un cuarto de él.

In [6]:
# Mayor deriva relativa de S exp(R0 R / N) a lo largo de la corrida del motor, con dos pasos, para el caso de epidemia
p = params_of("EX02_epidemic")
r0 = p.beta / p.gamma
for h in (0.25, 0.0625):
    r = simulate(p, dt=h)
    invariant = np.array(r.S) * np.exp(r0 * np.array(r.R) / p.N)
    drift = float(np.max(np.abs(invariant / invariant[0] - 1.0)))
    print(f"dt={h:6.4f}  mayor deriva relativa={drift:.5f}")

dt=0.2500  mayor deriva relativa=0.03143
dt=0.0625  mayor deriva relativa=0.00772


Ahora se resuelve la relación derivada, con la infección inicial, con el mismo `final_size` de `data-pipeline/pipeline/oracles.py` (su segundo argumento es la fracción inicial de infectados), para cada caso con $R_0 > 1$, y se ponen tres números lado a lado: la relación del nivel 2, la derivada y la tasa de ataque del motor con su paso. El caso de propagación lenta se marca, porque su brote no terminó en el horizonte.

In [7]:
# Resolver 1 - z = (1 - i0) exp(-R0 z) y ponerla junto a la relación del nivel 2 y a la tasa de ataque del motor
print(f"{'case':18s} {'nivel 2':>9s} {'derivada':>9s} {'motor':>9s}  terminó en el horizonte")
for case_id in SUPERCRITICAL:
    q = params_of(case_id)
    r = simulate(q)
    ended = r.I[-1] / q.N < 1e-6
    print(f"{case_id:18s} {final_size(q.beta / q.gamma):9.5f} {final_size(q.beta / q.gamma, q.I0 / q.N):9.5f} {r.attack_rate:9.5f}  {ended}")

case                 nivel 2  derivada     motor  terminó en el horizonte
EX02_epidemic        0.92044   0.92049   0.92365  True
EX03_fast_burn       0.99748   0.99749   0.99821  True
EX04_slow_spread     0.31370   0.31563   0.29928  False


La brecha que queda entre la relación derivada y el motor debería ser el error de primer orden del paso. Si el error es $c \, h$ para un paso $h$, entonces $2 A(h/2) - A(h)$ lo cancela, donde $A(h)$ es la tasa de ataque con paso $h$ (Hairer, Norsett y Wanner, 1993, dan el primer orden). La celda lo aplica al caso de epidemia, desde los pasos 1/4, 1/8 y 1/16 de día, y lo compara con la relación derivada.

In [8]:
# Quitar el error de primer orden del paso por extrapolación y comparar con el tamaño final derivado
p = params_of("EX02_epidemic")
attack = {h: simulate(p, dt=h).attack_rate for h in (0.25, 0.125, 0.0625)}
extrapolated = [2 * attack[0.125] - attack[0.25], 2 * attack[0.0625] - attack[0.125]]
target = final_size(p.beta / p.gamma, p.I0 / p.N)
print("tamaño final derivado:", round(target, 6))
for h, value in attack.items():
    print(f"dt={h:6.4f}  motor={value:.6f}  brecha={value - target:+.6f}")
print("extrapolado (0,25 y 0,125):", round(extrapolated[0], 6), " (0,125 y 0,0625):", round(extrapolated[1], 6))

tamaño final derivado: 0.920487
dt=0.2500  motor=0.923654  brecha=+0.003167
dt=0.1250  motor=0.922060  brecha=+0.001572
dt=0.0625  motor=0.921271  brecha=+0.000783
extrapolado (0,25 y 0,125): 0.920465  (0,125 y 0,0625): 0.920482


## Resultados

- La matriz de la siguiente generación da el mismo $R_0$ que $\beta/\gamma$ en cada caso, como debe ser con un compartimento infectado, y el signo de la tasa de crecimiento del compartimento infectado ($\beta - \gamma$) coincide con el lado de 1 en que está $R_0$: es la condición de estabilidad de van den Driessche y Watmough (2002). El crecimiento medido por el motor en dos días queda algo bajo $\beta - \gamma$, porque la reserva de susceptibles ya empezó a reducirse. Lo que agrega la matriz es generalidad: para un modelo con varios compartimentos infectados las mismas dos líneas (construir $F$ y $V$, tomar el radio espectral) dan $R_0$, donde $\beta/\gamma$ no tiene equivalente. Este modelo no lo necesita, y el motor no lo implementa.
- La cantidad $S \, e^{R_0 R / N}$ es constante en la solución exacta y deriva a lo largo de la corrida del motor unos pocos por ciento con el paso del motor; con un cuarto del paso la deriva es unas cuatro veces menor. La derivación se sostiene, y la deriva es el error de Euler hacia adelante.
- La relación derivada, con la infección inicial, difiere del nivel 2 en 0,00005 en el caso de epidemia y en 0,000002 en el de combustión rápida. La tasa de ataque del motor queda sobre ambas y se acerca al valor derivado al reducir el paso, y los valores extrapolados coinciden con él dentro de 0,00003 (dentro de 0,00001 con el par de pasos más fino). Así que la brecha vista en el nivel 2 es el error de primer orden del paso de un cuarto de día del motor, no un defecto de la relación. Para el caso de propagación lenta el brote no terminó, así que ninguna de las relaciones aplica aún a su tasa de ataque.
- Lo que estas corridas no muestran es nada sobre modelos con más compartimentos: el motor de ejemplo tiene solo tres.

## Resumen

- Estado del arte 1: $R_0$ como radio espectral de $F V^{-1}$ se reduce aquí a $\beta/\gamma$ y se generaliza a modelos mayores.
- Estado del arte 2: el tamaño final derivado de las ecuaciones, con la infección inicial, explica el pequeño exceso del motor sobre la relación del nivel 2 como error de Euler hacia adelante.
- Ambos coinciden con el nivel 2 en los mismos casos; ninguno cambia el motor.

## Referencias

- van den Driessche, P. and Watmough, J. (2002). Reproduction numbers and sub-threshold endemic equilibria for compartmental models of disease transmission. Mathematical Biosciences. https://doi.org/10.1016/S0025-5564(02)00108-6
- Miller, J. C. (2012). A note on the derivation of epidemic final sizes. Bulletin of Mathematical Biology. https://doi.org/10.1007/s11538-012-9749-6
- Kermack, W. O. and McKendrick, A. G. (1927). A contribution to the mathematical theory of epidemics. Proceedings of the Royal Society A. https://doi.org/10.1098/rspa.1927.0118
- Hairer, E., Norsett, S. P. and Wanner, G. (1993). Solving Ordinary Differential Equations I: Nonstiff Problems. Springer. https://doi.org/10.1007/978-3-540-78862-1
- NumPy, https://pypi.org/project/numpy/ (pinned in `notebooks/requirements.txt`).
- Matplotlib, https://pypi.org/project/matplotlib/ (pinned in `notebooks/requirements.txt`).
- El motor y los casos de este repositorio, https://github.com/fsantibanezleal/CAOS_PRODUCT_TEMPLATE (`data-pipeline/pipeline/model/sir.py`).

Lo que se atribuye a cada obra es lo que registra la base de investigación del SDD (`docs/design/SDD.md`); el texto de sus páginas no se volvió a leer, solo sus registros de Crossref.